# Data Quality Checks (YAML Config-Driven)

**Purpose:** Generic, automated data quality checks that protect downstream reports from bad data.

**How it works:**
1. Accepts a `table_name` parameter from the job
2. Reads validation rules from a YAML config file (`config/{table_name}.yml`)
3. Runs checks by category:
   - **Freshness** — is new data arriving on schedule?
   - **Completeness** — expected row count bounds (rolling weekly windows vs. historical baseline)
   - **Correctness** — not_null, no_future_date, range, unique, valid_values
4. Applies severity-based actions:
   - **Critical** = halt the job + alert immediately
   - **Warning** = alert but don't halt
5. Logs all results to `raw_data_quality_check_log` for dashboards and trend analysis

**Adding checks for a new table:** Create a new YAML file in the `config/` folder. No code changes needed.

In [0]:
import yaml
import os
import json
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, TimestampType, FloatType

param_table_name = dbutils.widgets.get("table_name")
param_ops_catalog = dbutils.widgets.get("ops_catalog")

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(f"/Workspace{notebook_path}")
CONFIG_DIR = os.path.normpath(os.path.join(notebook_dir, "../../config"))
CONFIG_FILE = os.path.join(CONFIG_DIR, f"{param_table_name}.yml")

LOG_TABLE = "dev_operations.manufacturing_staging.raw_data_quality_check_log"


print(f"Config file: {CONFIG_FILE}")
print(f"Log table: {LOG_TABLE}")

In [0]:
def get_last_metric(check_name):
    """Look up the most recent metric_value from the log table for a given check."""
    row = (
        spark.table(LOG_TABLE)
        .filter(
            (F.col("table_name") == FULLY_QUALIFIED_TABLE)
            & (F.col("check_name") == check_name)
            & (F.col("metric_value").isNotNull())
        )
        .orderBy(F.col("run_timestamp").desc())
        .limit(1)
        .select("metric_value")
        .first()
    )
    return row["metric_value"] if row and row["metric_value"] else None


def log_result(check, description, failed_rows, passed, severity, column, metric_value=None):
    """Append a check result to the results list in a standard format."""
    results.append({
        "check": check,
        "description": description,
        "failed_rows": failed_rows,
        "passed": passed,
        "severity": severity,
        "column": column,
        "metric_value": metric_value,
    })


def load_active_suppressions(table_fqn, suppression_table):
    """Load all active, non-expired suppressions for a given table."""
    try:
        return (
            spark.table(suppression_table)
            .filter(
                (F.col("table_name") == table_fqn)
                & (F.col("active") == True)
                & ((F.col("expires_at").isNull()) | (F.col("expires_at") > F.current_timestamp()))
            )
            .select("suppression_key", "column_name", "primary_key_json")
            .collect()
        )
    except Exception:
        return []


def was_previously_passing(table_fqn, check_name, before_ts):
    """Return True if the previous run passed (freshness transition detection)."""
    row = (
        spark.table(LOG_TABLE)
        .filter(
            (F.col("table_name") == table_fqn)
            & (F.col("check_name") == check_name)
            & (F.col("run_timestamp") < F.lit(before_ts))
        )
        .orderBy(F.col("run_timestamp").desc())
        .limit(1)
        .select("passed")
        .first()
    )
    return row["passed"] if row else True


def is_suppressed(failure, active_suppressions):
    """Return True if all failing keys are covered by suppression rows."""
    failing_keys = failure.get("failing_keys", [])
    failing_pk_rows = failure.get("failing_pk_rows", [])
    column = failure.get("column", "")
    if not failing_keys:
        return False

    col_suppressions = [r for r in active_suppressions if r["column_name"] == column]

    def pk_matches(suppression_json, pk_rows):
        try:
            suppressed_pk = json.loads(suppression_json)
        except (json.JSONDecodeError, TypeError):
            return False
        return any(
            all(str(pk_row.get(col)) == str(val) for col, val in suppressed_pk.items())
            for pk_row in pk_rows
        )

    def key_is_covered(key):
        for row in col_suppressions:
            if row["suppression_key"] != key:
                continue
            if row["primary_key_json"] is None:
                return True
            if failing_pk_rows and pk_matches(row["primary_key_json"], failing_pk_rows):
                return True
        return False

    return all(key_is_covered(k) for k in failing_keys)


def filter_alertable(results, table_fqn, active_suppressions, run_ts):
    """Split failures into alertable vs suppressed. Returns (alertable, suppressed_names)."""
    alertable = []
    suppressed = []
    for f in [r for r in results if not r["passed"]]:
        if "freshness" in f["check"]:
            if not was_previously_passing(table_fqn, f["check"], run_ts):
                suppressed.append(f["check"])
                continue
        else:
            if is_suppressed(f, active_suppressions):
                suppressed.append(f["check"])
                continue
        alertable.append(f)
    return alertable, suppressed

In [0]:
if not os.path.exists(CONFIG_FILE):
    raise FileNotFoundError(
        f"No config file found at {CONFIG_FILE}. "
        f"Create a YAML config for '{param_table_name}' in the config/ folder."
    )

with open(CONFIG_FILE, "r") as f:
    config = yaml.safe_load(f)

# Build the full table name
catalog_name = param_ops_catalog
schema_name = config["schema"]
table_name = config["table"]
FULLY_QUALIFIED_TABLE = f"{catalog_name}.{schema_name}.{table_name}"

# Primary key column(s) — used to surface failing row IDs in email alerts and suppressions.
_pk_raw = config.get("primary_key")
if isinstance(_pk_raw, str):
    primary_key_cols = [_pk_raw] if _pk_raw else []
elif isinstance(_pk_raw, list):
    primary_key_cols = _pk_raw
else:
    primary_key_cols = []
primary_key = primary_key_cols[0] if primary_key_cols else None

print(f"Data quality checks for: {FULLY_QUALIFIED_TABLE}")
print(f"Config loaded from: {CONFIG_FILE}")
print(f"Primary key columns: {primary_key_cols or 'not configured'}")

In [0]:
# Load the table into a DataFrame so we can run checks against it
df = spark.table(FULLY_QUALIFIED_TABLE)
total_rows = df.count()
results = []  # check results stored
today = date.today()

# Date column — used for freshness checks only.
date_col = config.get("date_column")
if not date_col:
    raise ValueError(
        f"'date_column' is required in the YAML config for '{FULLY_QUALIFIED_TABLE}'. "
        f"No fallback detection is used — specify the exact date column for freshness."
    )

# Latest date in the table — passed to email alerts as "Data refreshed"
data_refresh_date = str(df.agg(F.max(F.col(date_col))).first()[0] or "N/A")

# Parse cron schedule for both cadence and SLA time.
cron_expression = config.get("freshness", {}).get("cron_expression", "0 0 0 * * *")
cron_parts      = cron_expression.split()
dow_field       = cron_parts[5] if len(cron_parts) >= 6 else "*"
is_daily        = dow_field in ("*", "?")

# SLA time comes from the cron's hour/minute fields.
try:
    sla_hour   = int(cron_parts[2])
    sla_minute = int(cron_parts[1])
    sla_time   = datetime.now().replace(hour=sla_hour, minute=sla_minute, second=0, microsecond=0)
except (IndexError, ValueError):
    sla_time   = None  # no SLA enforcement if cron can't be parsed

## 1. Freshness
> Is new data arriving on schedule? If not, skip the SD check.

In [0]:
freshness_passed = True

if "freshness" in config:
    freshness_config  = config["freshness"]
    expected_lag_days = freshness_config.get("expected_lag_days", 0)

    # Only flag staleness if we are past the pipeline's scheduled SLA time.
    # Before the SLA, data may not have loaded yet — don't alert.
    now      = datetime.now()
    past_sla = sla_time is None or now >= sla_time

    current_max_date = df.select(F.max(F.to_date(F.col(date_col)))).first()[0]

    if current_max_date:
        expected_date = today - timedelta(days=expected_lag_days if is_daily else 7 + expected_lag_days)
        data_stale    = current_max_date < expected_date
        freq          = "daily" if is_daily else "weekly"
        sla_note      = f", SLA: {sla_time.strftime('%H:%M')}" if sla_time else ""

        if data_stale and not past_sla:
            freshness_passed = True
            description = (f"Pre-SLA. max({date_col}) = {current_max_date} — "
                           f"pipeline hasn't run yet ({freq}, lag={expected_lag_days}d{sla_note}).")
        elif data_stale:
            freshness_passed = False
            description = (f"STALE. max({date_col}) = {current_max_date} "
                           f"(expected >= {expected_date}, {freq}, lag={expected_lag_days}d, SLA missed{sla_note}).")
        else:
            freshness_passed = True
            description = (f"Fresh. max({date_col}) = {current_max_date} "
                           f"(expected >= {expected_date}, {freq}, lag={expected_lag_days}d{sla_note}).")
    else:
        freshness_passed = False
        description = f"STALE. No date found in {date_col}."

    results.append({
        "check":        "freshness",
        "description":  description,
        "failed_rows":  0 if freshness_passed else 1,
        "passed":       freshness_passed,
        "severity":     freshness_config["severity"],
        "column":       date_col,
        "metric_value": str(current_max_date) if current_max_date else None
    })

## 2. Completeness
> Is the expected volume of data present? Checks rolling weekly windows against historical baseline bounds.

In [0]:
from datetime import date, timedelta

AGGREGATE_FN = {
    "count_distinct": lambda col: F.countDistinct(col),
    "count": lambda col: F.count(col),
    "sum": lambda col: F.sum(col),
}

for check in config.get("completeness", []):
    col         = check["column"]
    agg_type    = check.get("aggregate", "count_distinct")
    window_days = check.get("window_days", 7)
    baseline_d  = check.get("baseline_days", 183)
    lookback_d  = check.get("lookback_days", 366)
    sigma_n     = check.get("threshold_sigma", 3)
    severity    = check["severity"]
    check_name  = check.get("name", f"completeness_{col}")

    cutoff = today - timedelta(days=lookback_d)
    baseline_cutoff = today - timedelta(days=baseline_d)
    agg_fn = AGGREGATE_FN.get(agg_type, AGGREGATE_FN["count_distinct"])

    daily_counts = (
        df.filter(F.col(date_col) >= F.lit(cutoff))
        .groupBy(F.col(date_col).cast("date").alias("day"))
        .agg(agg_fn(col).alias("cnt"))
    )

    spine = spark.sql(f"SELECT explode(sequence(date'{cutoff}', current_date(), INTERVAL 1 DAY)) AS day")
    daily = (
        spine.join(daily_counts, "day", "left")
        .select("day", F.coalesce(F.col("cnt"), F.lit(0)).alias("cnt"))
    )

    w = Window.orderBy("day").rowsBetween(-(window_days - 1), 0)
    rolling = daily.select(
        "day",
        F.sum("cnt").over(w).alias("wk"),
        F.count("*").over(w).alias("n"),
        (F.col("day") >= F.lit(baseline_cutoff)).alias("is_recent"),
    )

    bounds_row = (
        rolling.filter((F.col("n") == window_days) & (~F.col("is_recent")))
        .agg(F.avg("wk").alias("mu"), F.stddev("wk").alias("sigma"), F.count("*").alias("n_win"))
        .first()
    )

    mu, sigma, n_win = bounds_row["mu"], bounds_row["sigma"], bounds_row["n_win"]

    if n_win and n_win >= window_days and sigma and sigma > 0:
        lower = mu - sigma_n * sigma
        upper = mu + sigma_n * sigma
        outlier_count = (
            rolling.filter(
                (F.col("n") == window_days) & F.col("is_recent")
                & ((F.col("wk") < lower) | (F.col("wk") > upper))
            ).count()
        )
        failed = outlier_count > 0
        description = (
            f"{outlier_count} weeks outside {sigma_n}\u03C3 bounds [{int(lower)}, {int(upper)}]"
            if failed else f"{check_name}: PASS"
        )
    else:
        outlier_count = 0
        failed = False
        description = f"{check_name}: PASS (insufficient baseline: {n_win} windows)"

    results.append({
        "check":        f"completeness_{check_name}",
        "description":  description,
        "failed_rows":  outlier_count,
        "passed":       not failed,
        "severity":     severity,
        "column":       col,
        "metric_value": str(outlier_count) if failed else None,
        "failing_keys": [],
    })

## 3. Correctness
> Do individual rows make sense? (nulls, future dates, out-of-range values, new extremes)

In [0]:

for check in config.get("correctness", []):
    check_type  = check.get("check_type", "sql")
    column_name = check.get("column", "")
    severity    = check["severity"]
    check_name  = check.get("name", column_name)

    if check_type != "sql":
        results.append({"check": f"{check_type}_{check_name}",
                        "description": f"Unsupported check_type '{check_type}' — convert to sql",
                        "failed_rows": 0, "passed": True, "severity": severity,
                        "column": column_name, "metric_value": None, "failing_keys": []})
        continue

    query_template = check.get("query")
    if not query_template:
        results.append({"check": f"sql_{check_name}",
                        "description": f"sql check '{check_name}' skipped — no query configured",
                        "failed_rows": 0, "passed": True, "severity": severity,
                        "column": column_name, "metric_value": None, "failing_keys": []})
        continue

    query = query_template.format(
        table_fqn=FULLY_QUALIFIED_TABLE,
        date_col=date_col or "",
    )
    fail_df     = spark.sql(query)
    failed_rows = fail_df.count()
    failing_keys = []
    failing_pk_rows = []
    metric_value = None
    detail = None

    if failed_rows > 0:
        # Collect sample rows once — derive column names from Row to avoid Analyze RPC
        sample_rows = fail_df.limit(10).collect()
        fail_cols = list(sample_rows[0].asDict().keys()) if sample_rows else []
        first_row = sample_rows[0] if sample_rows else None
        # Extract optional _metric_value and _description from SQL results
        if "_metric_value" in fail_cols and first_row:
            metric_value = str(first_row["_metric_value"])
        if "_description" in fail_cols and first_row:
            detail = str(first_row["_description"])
        elif first_row:
            # Build informative description from available columns
            bound_cols = [c for c in fail_cols if "bound" in c or "fence" in c]
            value_cols = [c for c in fail_cols if c.startswith("daily_") or c == column_name]
            sample_val = first_row[value_cols[0]] if value_cols else None
            bound_val = first_row[bound_cols[0]] if bound_cols else None
            parts = [f"{check_name}: FAIL — {failed_rows} row(s)"]
            if sample_val is not None and bound_val is not None:
                direction = "above" if sample_val > bound_val else "below"
                parts.append(f"value={sample_val:.2f} {direction} {bound_cols[0]}={bound_val:.2f}")
            elif sample_val is not None:
                parts.append(f"value={sample_val}")
            detail = ". ".join(parts)
        # Identify failing keys for email
        available_pk_cols = [c for c in primary_key_cols if c in fail_cols]
        # Fall back to date_col or first column if PK not in results
        id_cols = available_pk_cols or ([date_col] if date_col in fail_cols else fail_cols[:1])
        if id_cols:
            failing_keys = [str(r[id_cols[0]]) for r in sample_rows if id_cols[0] in fail_cols]
            failing_pk_rows = [{col: str(r[col]) for col in id_cols} for r in sample_rows]

    description = detail or (f"{check_name}: PASS" if failed_rows == 0
                             else f"{check_name}: FAIL — {failed_rows} failing rows")
    results.append({"check": f"sql_{check_name}", "description": description,
                    "failed_rows": failed_rows, "passed": (failed_rows == 0), "severity": severity,
                    "column": column_name, "metric_value": metric_value,
                    "failing_keys": failing_keys, "failing_pk_rows": failing_pk_rows})

## 4. Output
> Print summary, log results, and halt on critical failures.

In [0]:
print(f"\nResults for {FULLY_QUALIFIED_TABLE} ({total_rows:,} rows):")
for r in results:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"  [{r['severity'].upper()}] {status} | {r['check']} | {r['failed_rows']:,} failed | {r['description']}")

In [0]:
log_schema = StructType([
    StructField("run_timestamp", TimestampType(), False),
    StructField("table_name", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("check_description", StringType(), False),
    StructField("failed_row_count", IntegerType(), False),
    StructField("passed", BooleanType(), False),
    StructField("severity", StringType(), False),
    StructField("metric_value", StringType(), True),
    StructField("failing_keys", StringType(), True),  # JSON array of failing primary keys / dates / groups
])

from zoneinfo import ZoneInfo
run_ts = datetime.now(tz=ZoneInfo("America/Chicago")).replace(tzinfo=None)
log_rows = [
    (
        run_ts,
        FULLY_QUALIFIED_TABLE,
        r["check"],
        r["description"],
        r["failed_rows"],
        r["passed"],
        r["severity"],
        str(r["metric_value"]) if r.get("metric_value") is not None else None,
        json.dumps(r["failing_keys"]) if r.get("failing_keys") else None
    )
    for r in results
]

log_df = spark.createDataFrame(log_rows, schema=log_schema)
log_df.write.mode("append").option("mergeSchema", "true").saveAsTable(LOG_TABLE)

print(f"Results logged to {LOG_TABLE}")
display(log_df)

In [0]:
SUPPRESSION_TABLE = f"{param_ops_catalog}.manufacturing_staging.raw_dq_suppressions"
active_suppressions = load_active_suppressions(FULLY_QUALIFIED_TABLE, SUPPRESSION_TABLE)
alertable_failures, suppressed_checks = filter_alertable(results, FULLY_QUALIFIED_TABLE, active_suppressions, run_ts)

if suppressed_checks:
    print(f"Suppressed (not alerting): {suppressed_checks}")

if alertable_failures:
    email_payload = [
        {"check": f["check"], "severity": f["severity"],
         "failed_rows": f["failed_rows"], "description": f["description"],
         "failing_keys": f.get("failing_keys", [])}
        for f in alertable_failures
    ]
    try:
        dbutils.notebook.run(
            "../utils/send_email_alert.py", timeout_seconds=60,
            arguments={"recipients": "addie.kochanik@milwaukeetool.com",
                       "table_name": FULLY_QUALIFIED_TABLE,
                       "failures_json": json.dumps(email_payload),
                       "data_refresh_date": data_refresh_date,
                       "ops_catalog": param_ops_catalog})
        print("Email alert sent.")
    except Exception as e:
        print(f"WARNING: Email alert failed to send: {e}")

critical_failures = [f for f in alertable_failures if f["severity"] == "critical"]
if critical_failures:
    for f in critical_failures:
        print(f"  CRITICAL: {f['check']}: {f['failed_rows']:,} issue(s) | {f['description']}")
    raise ValueError(
        f"Data quality gate FAILED for {FULLY_QUALIFIED_TABLE}. "
        f"{len(critical_failures)} critical check(s) failed. Halting downstream processing.")

warning_failures = [f for f in alertable_failures if f["severity"] == "warning"]
if warning_failures:
    print(f"{len(warning_failures)} warning(s) failed. Email sent. Continuing.")
else:
    print(f"All checks passed for {FULLY_QUALIFIED_TABLE}. Safe to continue.")